# K2 — the signed update path, end to end

**Checks (handoff A1–A5):** an installed **0.6.0** copy offers the new release, downloads it, verifies it, and swaps it in;
a tampered download is refused; a cut download ends in an error; an offline start shows no dialog; the **backup key** signs
something the app accepts.

**How it works:** the notebook downloads the old exe from the release page, checks its SHA-256 against the published
`SHA256SUMS.txt`, and puts a **fresh copy in a folder of its own for every check** (an update replaces the exe, so a copy is
single-use). You click in the app; the notebook reads the result off the folder (`.old`, `update_temp.exe`, the exe's hash).

**Before you start:** close the installed app and any other copy (they share one data folder). Run Jupyter as Administrator.
**Each step closes its app by itself** when it ends -- also when it fails or raises an error -- and first closes any copy an earlier step left running, so a check never looks at the wrong window.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "runbook" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find runbook above it
from runbook import common
from runbook.common import Recorder, capture, ask_text
import getpass, os
from runbook import updater
rec = Recorder("updater")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")

SLUG = "enjay27/resonance-stream"
OLD_TAG = "v0.6.0"
NEW_TAG = ask_text("NEW_TAG", "Release the old copy should be offered [v0.6.1]: ") or "v0.6.1"
BASE = f"https://github.com/{SLUG}/releases/download"
OLD_EXE, NEW_EXE = f"Resonance-Stream-{OLD_TAG}.exe", f"Resonance-Stream-{NEW_TAG}.exe"
RUNS = common.RUNS / "updater"

## 0 · Get the old exe and the published hashes

In [ ]:
sums_old = updater.parse_sums(common.fetch_text(f"{BASE}/{OLD_TAG}/SHA256SUMS.txt", fixture=f"sums_{OLD_TAG}.txt"))
sums_new = updater.parse_sums(common.fetch_text(f"{BASE}/{NEW_TAG}/SHA256SUMS.txt", fixture=f"sums_{NEW_TAG}.txt"))
old_sha, new_sha = sums_old[OLD_EXE], sums_new[NEW_EXE]
old_path = updater.download(f"{BASE}/{OLD_TAG}/{OLD_EXE}", common.RUNS / "downloads" / OLD_EXE, old_sha,
                            fixture=f"fake_app_{OLD_TAG}.bin")
rec.auto("K2-0", f"{OLD_EXE} downloaded, its hash is the published one", True, old_sha[:16])
print("old exe :", old_path)
print("new hash:", new_sha)

def show_copies():
    print("\nWhere every copy folder stands now (exe hash / .old / update_temp.exe):")
    for r in updater.scan_copies(RUNS):
        print(f"  {r['folder']:6} exe {r['sha']:12}  .old={'yes' if r['old'] else 'no ':3}  temp={'yes' if r['temp'] else 'no'}")
    print(f"  (the OLD exe's hash starts {old_sha[:12]}, the NEW one's {new_sha[:12]})")

## Prepare · the app must be "set up" to offer an update
The update dialog only appears in an app that has **finished the first-run setup** (`init_done: true` in `config.json`).
A fresh copy with no config shows the setup wizard instead, which never offers an app update -- so A1/A2 cannot work on it, and
a "pass" for A3/A4 on the wizard screen means nothing. This step checks the config of the app's data folder and, **only if you type
`yes`**, makes the app skip the wizard (a backup of any existing file is kept; the last step offers to put everything back).
It does not download the model; translation stays off.

In [ ]:
cfg = updater.app_config_path()
state = updater.setup_state(cfg)
print("config file:", cfg)
print("state      :", state, "(ready = setup done; missing / not_done = the wizard would show)")
setup_undo = None
if state == "ready":
    rec.auto("K2-prep", "the app is set up (init_done), so it can offer an update", True, "config.json says init_done: true")
elif ask_text("SETUP_OK", "Type yes to mark setup done in config.json (backup kept, undo offered at the end): ").lower() == "yes":
    setup_undo = updater.mark_setup_done(cfg)
    rec.auto("K2-prep", "config.json marked init_done (backup kept, undone at the end)", True, f"was: {state}")
else:
    rec.record("K2-prep", "the app is set up for the update checks", "skip",
               "declined: a fresh copy shows the wizard and cannot offer the update")

## A1 · Signed update, end to end

In [ ]:
exe1 = updater.fresh_copy(old_path, RUNS, "A1")
with updater.step(rec, "A1", exe1.parent, strays=RUNS):
    pid = updater.launch(exe1)
    print("Started THIS copy for you:", exe1, f"(pid {pid})")
    row = rec.manual("A1-ui", "the old copy offers the new version, downloads it, and restarts into it",
        "The copy above is running now (the notebook started it, so it is the right one; copies left over from earlier steps were closed).\n"
        "1. The update dialog appears with Korean notes.  2. Start the update; watch the bar move.\n"
        "3. '다운로드 완료' must come only AFTER the bar is full and a moment of checking.  4. Press 재시작.\n"
        "5. The app restarts; its title bar says 0.6.1 (or the new version).  6. Answer here -- the notebook closes the app for you.",
        "dialog with notes -> bar -> 다운로드 완료 after the check -> restarts on the new version")
    if common.dry_run() and row.status != "skip":
        updater.simulate_app(os.environ.get("RUNBOOK_SIM_A1", "updated"), exe1, NEW_TAG)
    if row.status == "skip":
        for title in ("the exe is now the published new version", f"{exe1.name}.old is left, and is the old version",
                      "update_temp.exe is gone"):
            rec.record("A1", title, "skip", "not checked: the step above was skipped")
    else:
        checks = updater.check_a1(exe1.parent, exe1.name, old_sha, new_sha)
        for title, ok, evidence in checks:
            rec.auto("A1", title, ok, evidence)
        if not all(ok for _, ok, _ in checks):
            show_copies()

## A2 · Verify before the swap (a tampered download is refused)

In [ ]:
exe2 = updater.fresh_copy(old_path, RUNS, "A2")
with updater.step(rec, "A2", exe2.parent, strays=RUNS):
    pid = updater.launch(exe2)
    print("Started THIS copy for you:", exe2, f"(pid {pid})")
    row = rec.manual("A2-1", "download until '다운로드 완료' -- and STOP there",
        "The copy above is running now. Start the update. When '다운로드 완료' shows, DO NOT press 재시작. Leave the app running.\n"
        "Answer `pass` as soon as the dialog says 다운로드 완료 (`skip` if it never did).",
        "다운로드 완료 is showing, the app still runs the old version")
    if row.status == "pass":
        if common.dry_run():
            updater.simulate_app("downloaded", exe2, NEW_TAG)
        # is the app that is running really THIS copy? (the update lands in the folder of the exe that runs)
        import json
        if common.dry_run():
            text = json.dumps([{"Path": str(exe2 if os.environ.get("RUNBOOK_SIM_RUNNING") != "other" else exe1)}])
        else:
            text = capture(["powershell", "-NoProfile", "-Command", updater.POWERSHELL_PROCESSES])[1]
        running = updater.parse_process_paths(text)
        # pass only when the A2 copy runs AND nothing else does (the step closed earlier copies; a leftover would be a finding)
        rec.auto("A2-which", "the running app is the A2 copy",
                 any(updater.same_file(p, exe2) for p in running) and all(updater.same_file(p, exe2) for p in running),
                 "; ".join(running) or "no Resonance-Stream process is running")
        temp = exe2.parent / "update_temp.exe"
        if temp.exists():
            offset, before, after = updater.flip_one_byte(temp)
            rec.auto("A2-flip", "update_temp.exe: one byte flipped", True, f"offset {offset}: {before:#04x} -> {after:#04x}")
        else:
            rec.auto("A2-flip", "update_temp.exe is next to the exe", False, f"not found in {exe2.parent}")
            show_copies()
    if row.status == "pass" and (exe2.parent / "update_temp.exe").exists():
        rec.manual("A2-2", "press 재시작 on the tampered update: refused, old version keeps running",
            "NOW press 재시작 in the app.",
            "an error step appears (the reason says the file is not signed by a trusted key); the app does NOT restart and "
            "still shows the old version")
        for title, ok, evidence in updater.check_a2(exe2.parent, exe2.name, old_sha):
            rec.auto("A2", title, ok, evidence)
    else:
        print("A2 skipped: no 다운로드 완료 (see above)")

In [ ]:
# (A2's second half is in the cell above: one step, one try / finally, so the app stays open until it is done)
print("A2 done -- the app was closed by the step.")

## A3 · A download that is cut
**(Earlier runs answered `pass` here on the setup-wizard screen: that does not count -- run it again now that setup is done.)**
**On 0.6.0 this check cannot be judged:** it has the 0%-hang bug (K1) that 0.6.1 fixed, so cutting the network may leave the
bar stuck instead of showing an error. Record what you see; answer `skip: hung at N%` if it hangs — that is expected here.
Judge A3 for real with a 0.6.1 copy updating to the next release.

In [ ]:
exe3 = updater.fresh_copy(old_path, RUNS, "A3")
with updater.step(rec, "A3", exe3.parent, strays=RUNS):
    pid = updater.launch(exe3)
    print("Started THIS copy for you:", exe3, f"(pid {pid})")
    row3 = rec.manual("A3", "cut the network mid-download (on 0.6.0 a hang is EXPECTED)",
        f"The copy above is running now. Start the update; when the bar is about halfway, turn Wi-Fi off / pull the cable. Wait ~1 minute.\n"
        "`pass` = an error step with a reason and a working 다시 시도.  `skip: hung at N%` = stuck bar (expected on 0.6.0).  "
        "`fail: <what>` = anything else.  Then answer here (the notebook closes the app) and turn the network back on.",
        "error with a reason + 다시 시도 (0.6.1+), or a stuck bar (0.6.0, K1)")
    if common.dry_run() and row3.status != "pass":
        updater.simulate_app("stalled" if row3.status == "skip" and "hung" in rec.rows[-1].evidence else "nothing", exe3, NEW_TAG)
    left = updater.leftovers(exe3.parent, exe3.name)
    if row3.status == "skip" and not left["temp"] and not left["part"]:
        rec.record("A3", "no half file named update_temp.exe", "skip", "not checked: nothing was downloaded")
    else:
        for title, ok, evidence in updater.check_a3(exe3.parent, exe3.name):
            rec.auto("A3", title, ok, evidence)

## A4 · Offline start

In [ ]:
exe4 = updater.fresh_copy(old_path, RUNS, "A4")
with updater.step(rec, "A4", exe4.parent, strays=RUNS):
    if not common.dry_run():
        input("Turn the network OFF now (Wi-Fi off / cable out), then press Enter here to start the copy... ")
    pid = updater.launch(exe4)
    print("Started THIS copy for you:", exe4, f"(pid {pid})")
    rec.manual("A4", "offline start: no update dialog, the system log says 'Check failed'",
        "The copy above is running, with the network off. Open the system tab. Then answer here (the notebook closes the app) and turn the network back on.",
        "the app starts, no update dialog, the system log has 'Check failed'")

## A5 · The backup key signs something the app accepts
Needs Node (`npx`) and Rust (`cargo`) on this PC, and the **backup key file + its password**. The password is typed into a
hidden prompt, put in this notebook's environment only for the signing, and removed afterwards; it is never printed or saved.

In [ ]:
key_path = ask_text("BACKUP_KEY", "Full path of the BACKUP key file (e.g. C:/keys/backup.key), then Enter: ")
password = os.environ.get("RUNBOOK_TEXT_BACKUP_PW", "") if common.dry_run() else getpass.getpass("Backup key password: ")
target = common.RUNS / "a5" / "sign-me.bin"
target.parent.mkdir(parents=True, exist_ok=True)
target.write_bytes(b"resonance-stream backup key check\n")
sig = Path(str(target) + ".sig")
sig.unlink(missing_ok=True)
try:
    # the key's CONTENTS go in TAURI_SIGNING_PRIVATE_KEY -- exactly what release.yml does
    key_text = "dry-run-key" if (common.dry_run() and key_path) else updater.read_key_file(key_path)
except ValueError as e:
    key_text = None
    rec.auto("A5-1", "the backup key file can be read", False, str(e))
if key_text is not None:
    os.environ["TAURI_SIGNING_PRIVATE_KEY"] = key_text
    os.environ["TAURI_SIGNING_PRIVATE_KEY_PASSWORD"] = password
    try:
        code, out = capture(["npx", "--yes", "@tauri-apps/cli@2", "signer", "sign", "--app-version", "0.6.0", str(target)],
                            fixture="tauri_sign_ok.txt", timeout=900)
    finally:
        os.environ.pop("TAURI_SIGNING_PRIVATE_KEY", None)
        os.environ.pop("TAURI_SIGNING_PRIVATE_KEY_PASSWORD", None)
        key_text = password = ""
    if common.dry_run():
        import shutil
        shutil.copy(common.FIXTURES / "fake_backup.sig", sig)
    rec.auto("A5-1", "the file was signed", code == 0 and sig.exists(),
             " ".join(out.split())[-200:] if code != 0 or not sig.exists() else "signature written")

In [ ]:
if sig.exists():
    key_id = updater.signature_key_id(sig.read_text())
    rec.auto("A5-2", "it was signed by the BACKUP key, not the primary", updater.is_backup_key(key_id), f"key id {key_id}")
    VERIFY = ["cargo", "run", "--release", "-p", "resonance-core", "--example", "verify_update", "--"]
    code, out = capture(VERIFY + [str(target), str(sig), "0.6.0"], fixture="verify_update_ok.txt", timeout=1800)
    ok, evidence = updater.parse_verify(code, out, "0.6.0")
    rec.auto("A5-3", "the app's own check accepts it for 0.6.0", ok, evidence)
    code, out = capture(VERIFY + [str(target), str(sig), "0.6.9"], fixture="verify_update_refused.txt", timeout=1800)
    ok, evidence = updater.parse_verify(code, out, "0.6.9")
    rec.auto("A5-4", "...and refuses it for another version (0.6.9)", not ok, evidence)
else:
    print("A5 stopped: no signature file")

## Put the config back
Close every copy of the app first (a running app would overwrite the file again).

In [ ]:
closed = updater.stop_copies(RUNS)
if closed:
    print("Closed copies still running:", closed)
if setup_undo is not None:
    if ask_text("RESTORE", "Put config.json back exactly as it was before this notebook? (yes / no): ").lower() == "yes":
        updater.undo_setup(setup_undo)
        print("config.json restored.")
    else:
        print("Kept. The original is saved next to it as config.json" + updater.BACKUP_SUFFIX + " (if there was one).")
else:
    print("Nothing to put back.")

## Report
Paste the output of the next cell to Claude.

In [ ]:
print(rec.report())